In [ ]:


import pandas as pd
import numpy as np
import joblib
import re
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# FEATURE ENGINEERING (doit être identique à failure_classification.py)
# ─────────────────────────────────────────────────────────────

def build_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df.copy()

    X["maintenance_stress"] = (
        X["km_since_oil_change"] / (X["km_since_brake_service"] + 1)
    )
    X["thermal_stress"] = (
        X["avg_coolant_temp_30d"] * X["avg_rpm_30d"] / 1000
    )
    X["thermal_overload_ratio"] = (
        X["max_coolant_temp_30d"] / (X["avg_coolant_temp_30d"] + 1e-6)
    )
    X["voltage_instability"] = (
        (X["avg_voltage_30d"] - X["min_voltage_30d"])
        / (X["avg_voltage_30d"] + 1e-6)
    )
    X["dtc_acceleration"] = (
        (X["dtc_count_7d"] * 4 - X["dtc_count_30d"]).clip(lower=0)
    )
    X["dtc_density"] = (
        X["dtc_count_90d"] / (X["total_engine_hours"] + 1)
    )
    X["cumulative_wear"] = (
        X["mileage_km"] / 1000 * X["vehicle_age_years"]
    )
    X["engine_stress"] = (
        X["avg_rpm_30d"] * X["avg_throttle_30d"] * X["driving_aggression_score"]
    )
    X["maintenance_neglect"] = (
        X["km_since_oil_change"]
        + X["km_since_brake_service"]
        + X["km_since_battery_change"]
        + X["km_since_coolant_flush"]
    ) / 4

    return X


# ─────────────────────────────────────────────────────────────
# CHARGEMENT DES MODÈLES
# ─────────────────────────────────────────────────────────────

print("📦 Loading saved models...")

# --- Anomaly detection (anomaly_detection.py) ---
iso_model = joblib.load("iso_model.pkl")
scaler    = joblib.load("scaler.pkl")

# --- Risk models par composant (failure_classification.py) ---
risk_models   = joblib.load("risk_models.pkl")
risk_encoders = joblib.load("risk_encoders.pkl")
risk_imputers = joblib.load("risk_imputers.pkl")
risk_features = joblib.load("risk_features.pkl")

# --- Fault type models par composant (failure_classification.py) ---
fault_models   = joblib.load("fault_models.pkl")
fault_encoders = joblib.load("fault_encoders.pkl")
fault_imputers = joblib.load("fault_imputers.pkl")
fault_features = joblib.load("fault_features.pkl")

print("✅ All models loaded")

# ─────────────────────────────────────────────────────────────
# DATASET DE RECOMMANDATION (TF-IDF)
# ─────────────────────────────────────────────────────────────

print("\n📂 Loading recommendation dataset...")

rec_df = pd.read_csv("ML_Car_Diagnostic_Agent_AI_Assistant.csv")
rec_df.columns = rec_df.columns.str.strip().str.lower().str.replace(" ", "_")

rec_df = rec_df.dropna(subset=["component", "problem_description"])

rec_df["combined_text"] = (
    rec_df["component"].fillna("") + " "
    + rec_df["component"].fillna("") + " "
    + rec_df["problem_description"].fillna("") + " "
    + rec_df["problem_description"].fillna("") + " "
    + rec_df["diagnosis"].fillna("") + " "
    + rec_df["ecu_data"].fillna("") + " "
    + rec_df["service_history"].fillna("") + " "
    + rec_df["how_to_fix_the_problem"].fillna("") + " "
    + rec_df["solution_used"].fillna("")
)

RISK_MAP = {
    "low": 1,      "low_risk": 1,
    "medium": 2,   "medium_risk": 2,
    "high": 3,     "high_risk": 3,
    "critical": 4, "critical_risk": 4,
}
rec_df["risk_score_num"] = (
    rec_df["risk_level"].str.lower().str.strip().map(RISK_MAP).fillna(2)
)

print(f"✅ {len(rec_df)} historical cases loaded")

print("\n🔍 Building TF-IDF index...")

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10_000,
    sublinear_tf=True,
    min_df=1,
)
tfidf_matrix = tfidf.fit_transform(rec_df["combined_text"])

print(f"✅ TF-IDF matrix: {tfidf_matrix.shape}")


# ─────────────────────────────────────────────────────────────
# CONSTANTES & MAPPINGS SÉMANTIQUES
# ─────────────────────────────────────────────────────────────

OBD2_FEATURES = [
    "engine_rpm",
    "vehicle_speed_kmh",
    "coolant_temp_c",
    "intake_air_temp_c",
    "maf_airflow_gs",
    "throttle_position_pct",
    "control_module_voltage_v",
    "engine_load_pct",
    "short_fuel_trim_pct",
    "long_fuel_trim_pct",
]

# Ordre sémantique correct (≠ ordre alphabétique de LabelEncoder)
SEMANTIC_RISK_ORDER = {
    "low": 0,      "low_risk": 0,
    "medium": 1,   "medium_risk": 1,
    "high": 2,     "high_risk": 2,
    "critical": 3, "critical_risk": 3,
}

HIGH_RISK_LABELS = {"high_risk", "critical_risk", "high", "critical"}

RESULT_MAP = {
    "resolved": 1.0, "fixed": 1.0, "success": 1.0,
    "partial": 0.5,  "partially resolved": 0.5,
    "failed": 0.0,   "unresolved": 0.0,
}
REPAIR_STATUS_MAP = {
    "fixed": 1.0,    "completed": 1.0, "resolved": 1.0,
    "in progress": 0.5, "pending": 0.3,
    "failed": 0.0,   "unresolved": 0.0,
}

# Poids des urgences pour tri visuel
URGENCY_WEIGHT = {
    "🔴 CRITICAL": 4,
    "🟠 HIGH": 3,
    "🟡 MEDIUM": 2,
    "🟢 LOW": 1,
}


# ─────────────────────────────────────────────────────────────
# 1. ANOMALY DETECTION
# ─────────────────────────────────────────────────────────────

def get_anomaly_prob(obd2_row: dict) -> float:
    """
    Probabilité d'anomalie [0-1] via IsolationForest.
    1 = très anormal, 0 = normal.
    """
    df_row = pd.DataFrame([obd2_row])[OBD2_FEATURES]
    scaled = scaler.transform(df_row)
    score  = iso_model.decision_function(scaled)[0]
    # Sigmoïde inversée : score positif → normal → prob basse
    anomaly_prob = 1 / (1 + np.exp(score * 3))
    return float(np.clip(anomaly_prob, 0, 1))


# ─────────────────────────────────────────────────────────────
# 2. RISK LEVEL PAR COMPOSANT  (nouveaux modèles per-component)
# ─────────────────────────────────────────────────────────────

def _prepare_features(row: dict, feats: list, imputer) -> pd.DataFrame:
    """
    Construit et impute un DataFrame prêt pour prédiction.
    Utilise build_engineered_features pour calculer les features dérivées.
    """
    X = build_engineered_features(pd.DataFrame([row]))

    # Ajouter les colonnes manquantes
    for f in feats:
        if f not in X.columns:
            X[f] = np.nan

    X = X[feats]
    X = pd.DataFrame(imputer.transform(X), columns=feats)
    return X


def get_risk_level_and_proba(vehicle_data: dict):
    """
    Prédit le niveau de risque avec le modèle CatBoost spécifique
    au composant (risk_models.pkl).

    Returns
    -------
    risk_label : str   — ex. "high_risk"
    risk_proba : float — confiance [0-1]
    risk_index : int   — indice encodé
    all_probas : dict  — {classe: proba} pour toutes les classes
    """
    component = str(vehicle_data.get("component", "")).strip().lower()

    if component not in risk_models:
        print(f"  ⚠️  Aucun modèle risk pour '{component}' — retour 'medium_risk'")
        return "medium_risk", 0.5, 1, {}

    feats   = risk_features[component]
    imputer = risk_imputers[component]
    model   = risk_models[component]
    encoder = risk_encoders[component]

    X         = _prepare_features(vehicle_data, feats, imputer)
    pred_idx  = int(model.predict(X).flatten()[0])
    probas    = model.predict_proba(X)[0]

    risk_label = encoder.classes_[pred_idx]
    risk_proba = float(probas[pred_idx])
    all_probas = {cls: round(float(p), 4) for cls, p in zip(encoder.classes_, probas)}

    return risk_label, risk_proba, pred_idx, all_probas


# ─────────────────────────────────────────────────────────────
# 3. FAULT TYPE PAR COMPOSANT  (nouveaux modèles per-component)
# ─────────────────────────────────────────────────────────────

def get_fault_type_and_proba(vehicle_data: dict):
    """
    Prédit le type de panne avec le modèle CatBoost spécifique
    au composant (fault_models.pkl).

    Returns
    -------
    fault_label : str   — ex. "engine_overheating"
    fault_proba : float — confiance [0-1]
    all_probas  : dict  — {classe: proba} pour toutes les classes
    """
    component = str(vehicle_data.get("component", "")).strip().lower()

    if component not in fault_models:
        print(f"  ⚠️  Aucun modèle fault pour '{component}' — retour 'normal'")
        return "normal", 0.5, {}

    feats   = fault_features[component]
    imputer = fault_imputers[component]
    model   = fault_models[component]
    encoder = fault_encoders[component]

    X          = _prepare_features(vehicle_data, feats, imputer)
    pred_idx   = int(model.predict(X).flatten()[0])
    probas     = model.predict_proba(X)[0]

    fault_label = encoder.classes_[pred_idx]
    fault_proba = float(probas[pred_idx])
    all_probas  = {cls: round(float(p), 4) for cls, p in zip(encoder.classes_, probas)}

    return fault_label, fault_proba, all_probas


# ─────────────────────────────────────────────────────────────
# 4. ML RISK SCORE COMPOSITE
# ─────────────────────────────────────────────────────────────

def compute_ml_risk_score(
    anomaly_prob: float,
    risk_label: str,
    fault_label: str,
    fault_proba: float,
    n_risk_classes: int = 4,
) -> float:
    """
    ML Risk Score amélioré intégrant :
      - anomaly_prob  (IsolationForest)   → 35 %
      - risk_level    (modèle par comp.)  → 45 %
      - fault_proba   (confiance fault)   → 20 %

    Le fault_proba booste le score si une panne réelle est détectée
    (fault_label ≠ "normal"), et reste neutre sinon.
    """
    order = SEMANTIC_RISK_ORDER.get(risk_label.lower().strip(), 1)
    normalised_risk = order / max(n_risk_classes - 1, 1)

    # Contribution fault : pénalise uniquement si panne réelle détectée
    fault_contribution = fault_proba if fault_label.lower() != "normal" else 0.0

    ml_risk_score = (
        anomaly_prob     * 0.35
        + normalised_risk  * 0.45
        + fault_contribution * 0.20
    )
    return float(np.clip(ml_risk_score, 0, 1))


# ─────────────────────────────────────────────────────────────
# 5. TF-IDF SIMILARITY — enrichie par fault_label
# ─────────────────────────────────────────────────────────────

def get_similar_cases(
    component: str,
    problem_description: str,
    fault_label: str = "",
    ecu_data: str = "",
    service_history: str = "",
    top_k: int = 50,
) -> pd.DataFrame:
    """
    Récupère les cas historiques les plus proches via TF-IDF.
    Le fault_label prédit enrichit la requête pour cibler le bon type de panne.
    """
    # Le fault_label est ajouté en double poids dans la requête
    fault_hint = fault_label.replace("_", " ") if fault_label and fault_label != "normal" else ""

    query_parts = [
        component, component,
        problem_description, problem_description,
        fault_hint, fault_hint,        # nouveau : type de panne prédit
        ecu_data,
        service_history,
    ]
    query = " ".join(p for p in query_parts if p)

    query_vec = tfidf.transform([query])
    sims = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Priorité aux cas du même composant
    comp_lower = component.strip().lower()
    component_mask = rec_df["component"].str.lower().str.strip() == comp_lower
    if component_mask.sum() >= top_k:
        sims_filtered = sims.copy()
        sims_filtered[~component_mask.values] = -1.0
    else:
        sims_filtered = sims

    top_indices = np.argsort(sims_filtered)[::-1][:top_k]
    top_cases   = rec_df.iloc[top_indices].copy()
    top_cases["similarity_score"] = sims[top_indices]

    return top_cases.reset_index(drop=True)


# ─────────────────────────────────────────────────────────────
# 6. FUSION DES SCORES
# ─────────────────────────────────────────────────────────────

def encode_result(result_str) -> float:
    if pd.isna(result_str):
        return 0.5
    return RESULT_MAP.get(str(result_str).lower().strip(), 0.5)

def encode_repair_status(status_str) -> float:
    if pd.isna(status_str):
        return 0.5
    return REPAIR_STATUS_MAP.get(str(status_str).lower().strip(), 0.5)

def solution_specificity(solution_str) -> float:
    if pd.isna(solution_str):
        return 0.0
    return float(np.clip(len(str(solution_str).split()) / 8.0, 0.0, 1.0))

def problem_match_bonus(query_problem: str, case_problem: str) -> float:
    q_tokens = set(query_problem.lower().split())
    c_tokens = set(case_problem.lower().split())
    if not q_tokens:
        return 0.0
    return float(np.clip(len(q_tokens & c_tokens) / len(q_tokens), 0.0, 1.0))


# =============================================================
# FAULT → PROBLEM DESCRIPTION MAPPING (1-to-many)
# =============================================================
# Les fault_labels internes (snake_case) du modèle de classification
# ne correspondent PAS 1-pour-1 aux Problem Description du dataset
# de recommandation. Le dataset utilise un vocabulaire symptôme-centrique
# ("Brake noise", "Squealing brakes") alors que le modèle prédit des
# causes ("brake_wear", "brake_overheating").
#
# Ce mapping relie chaque cause interne à TOUTES les descriptions de
# symptômes associées dans le dataset, permettant un matching correct.
# À mettre à jour si de nouvelles valeurs apparaissent dans le dataset.
# =============================================================

FAULT_TO_PROBLEMS: dict[str, list[str]] = {

    # ── Engine ────────────────────────────────────────────────
    "engine_misfire": [
        "engine misfire",
        "rough idling",
        "engine hesitation",
        "engine surging",
        "misfiring engine",
    ],
    "engine_overheating": [
        "engine overheating",
        "overheating",
        "coolant temperature high",
        "engine running hot",
        "temperature warning light",
    ],
    "engine_stalling": [
        "engine stalling",
        "engine stalls",
        "engine cuts out",
        "sudden engine shutdown",
        "engine dies while driving",
    ],

    # ── Electrical ────────────────────────────────────────────
    "battery_drain_fast": [
        "battery drain",
        "battery draining fast",
        "dead battery",
        "battery keeps dying",
        "battery discharge",
    ],
    "wont_start": [
        "car won't start",
        "car does not start",
        "no crank no start",
        "engine won't crank",
        "starting problem",
    ],
    "headlights_flicker": [
        "headlights flickering",
        "headlights dim",
        "flickering lights",
        "headlight issues",
    ],
    "dashboard_flickering": [
        "dashboard flickering",
        "instrument cluster issues",
        "warning lights flickering",
        "gauge fluctuation",
    ],
    "power_window_flickering": [
        "power window not working",
        "window motor failure",
        "window regulator issue",
        "electric window problem",
    ],

    # ── Transmission ──────────────────────────────────────────
    "transmission_overheating": [
        "transmission overheating",
        "transmission running hot",
        "gearbox overheating",
        "transmission temperature high",
        "gear shift hard",           # symptôme fréquent de surchauffe
    ],
    "slipping_gears": [
        "slipping gears",
        "gear slipping",
        "transmission slipping",
        "delayed shifting",          # présent dans le dataset
        "gear grinding",             # présent dans le dataset
        "gear shift hard",
    ],

    # ── Battery ───────────────────────────────────────────────
    "battery_warning_light": [
        "battery warning light",
        "battery light on",
        "charging system warning",
        "alternator warning",
    ],
    "wont_start": [   # dupliqué volontairement : battery + electrical
        "car won't start",
        "dead battery",
        "no start",
    ],

    # ── Brakes ────────────────────────────────────────────────
    # Le dataset brakes contient : Soft brake pedal, Brake noise,
    # Squealing brakes, Vibration when braking, Brake pedal hard
    "brake_wear": [
        "brake noise",               # ← présent dans le dataset
        "squealing brakes",          # ← présent dans le dataset
        "vibration when braking",    # ← présent dans le dataset
        "brake wear",
        "grinding brakes",
        "brake pad worn",
    ],
    "brake_overheating": [
        "brake pedal hard",          # ← présent dans le dataset
        "soft brake pedal",          # ← présent dans le dataset (fade)
        "brake overheating",
        "brake fade",
        "burning smell brakes",
    ],
    "general_degradation": [
        "soft brake pedal",          # ← présent dans le dataset
        "vibration when braking",    # ← présent dans le dataset
        "general wear and tear",
        "brake pedal spongy",
        "brake performance loss",
    ],

    # ── Fuel system ───────────────────────────────────────────
    "injector_clogging": [
        "fuel injector clogging",
        "injector fouling",
        "poor fuel economy",
        "rough idle lean",
        "engine hesitation acceleration",
    ],
    "air_intake_leak_maf": [
        "air intake leak",
        "maf sensor fault",
        "mass airflow sensor",
        "rich mixture",
        "intake manifold leak",
    ],
    "fuel_pump_weakness": [
        "fuel pump failure",
        "fuel pump weak",
        "fuel pressure low",
        "engine sputtering",
        "fuel starvation",
    ],

    # ── Cooling system ────────────────────────────────────────
    "thermostat_failure": [
        "thermostat failure",
        "thermostat stuck",
        "coolant temperature fluctuation",
        "heater not working",
        "engine slow to warm up",
    ],
    "radiator_clogging": [
        "radiator clogging",
        "radiator blockage",
        "coolant flow restricted",
        "overheating highway",
        "coolant flush needed",
    ],
    "overheating": [
        "engine overheating",
        "overheating",
        "coolant boiling",
        "temperature gauge high",
        "steam from engine",
    ],
}

# Construit au chargement : index inversé problem_description (lower) → fault_labels
# Permet de retrouver rapidement tous les fault_labels associés à un problème.
_PROBLEM_TO_FAULTS: dict[str, list[str]] = {}
for _fault, _problems in FAULT_TO_PROBLEMS.items():
    for _prob in _problems:
        _PROBLEM_TO_FAULTS.setdefault(_prob.lower().strip(), []).append(_fault)

# Index rec_df : problem_description (lower) → indices de lignes
_PROBLEM_INDEX: dict = {}
def _build_problem_index():
    global _PROBLEM_INDEX
    for idx, row in rec_df.iterrows():
        key = str(row.get("problem_description", "")).lower().strip()
        _PROBLEM_INDEX.setdefault(key, []).append(idx)
_build_problem_index()


def fault_type_match_bonus(predicted_fault: str, case_problem_description: str) -> float:
    """
    Calcule la concordance entre le fault_label prédit et la problem_description
    d'un cas historique, en utilisant le mapping 1-to-many FAULT_TO_PROBLEMS.

    Logique :
      - On récupère la liste de toutes les descriptions associées au fault prédit.
      - On cherche une correspondance exacte ou partielle avec le cas historique.

    Score :
      1.0  — le cas a exactement l'une des descriptions associées au fault
      0.5  — correspondance partielle (≥ 50% des tokens en commun)
      0.0  — aucune correspondance ou fault == "normal"
    """
    if not predicted_fault or predicted_fault == "normal":
        return 0.0

    associated_problems = FAULT_TO_PROBLEMS.get(predicted_fault.lower(), [])
    if not associated_problems:
        # Fallback : underscores → espaces
        associated_problems = [predicted_fault.lower().replace("_", " ")]

    case_str = str(case_problem_description).lower().strip()

    # Correspondance exacte avec l'une des descriptions associées
    for prob in associated_problems:
        if prob.lower() == case_str:
            return 1.0

    # Correspondance partielle : meilleur ratio parmi toutes les descriptions
    best_overlap = 0.0
    c_tokens = set(case_str.split())
    for prob in associated_problems:
        t_tokens = set(prob.lower().split())
        if not t_tokens:
            continue
        overlap = len(t_tokens & c_tokens) / len(t_tokens)
        best_overlap = max(best_overlap, overlap)

    # Partiel plafonné à 0.5 pour ne pas surpasser le match exact
    return float(np.clip(best_overlap * 0.5, 0.0, 0.5))


def fuse_scores(
    similarity: float,
    ml_risk: float,
    case_result_score: float,
    repair_status_score: float,
    risk_alignment: float,
    specificity_score: float,
    risk_level_actuel: float,
    fault_match: float = 0.0,
) -> float:
    """
    Fusion pondérée — poids total = 1.0

      similarity          (0.35) TF-IDF match
      ml_risk             (0.20) score composite anomalie+risk+fault
      fault_match         (0.12) concordance type de panne prédit ↔ cas
      case_result_score   (0.10) outcome historique
      repair_status_score (0.09) statut réparation
      risk_alignment      (0.07) alignement niveau de risque
      specificity_score   (0.04) richesse de la solution
      risk_level_actuel   (0.03) urgence véhicule courant
    """
    final_score = (
        similarity          * 0.35
        + ml_risk           * 0.20
        + fault_match       * 0.12
        + case_result_score * 0.10
        + repair_status_score * 0.09
        + risk_alignment    * 0.07
        + specificity_score * 0.04
        + risk_level_actuel * 0.03
    )
    return float(np.clip(final_score, 0.0, 1.0))


# ─────────────────────────────────────────────────────────────
# 7. PIPELINE PRINCIPAL : recommend()
# ─────────────────────────────────────────────────────────────

def recommend(
    obd2_data: dict,
    vehicle_data: dict,
    component: str,
    problem_description: str,
    ecu_data: str = "",
    service_history: str = "",
    top_k: int = 5,
) -> dict:
    """
    Pipeline complet :
      1. Anomaly detection  → anomaly_prob         (IsolationForest)
      2. Risk classification → risk_label          (CatBoost par composant)
      3. Fault classification → fault_label        (CatBoost par composant)  ← NOUVEAU
      4. ML Risk Score composite                                               ← NOUVEAU
      5. TF-IDF similarity enrichie par fault_label                           ← NOUVEAU
      6. Score fusion avec fault_match bonus                                   ← NOUVEAU
      7. Déduplication + classement final

    Parameters
    ----------
    obd2_data         : dict — keys = OBD2_FEATURES
    vehicle_data      : dict — keys compatibles failure_classification
    component         : str  — ex. "engine", "brakes"
    problem_description : str
    ecu_data          : str  (optionnel)
    service_history   : str  (optionnel)
    top_k             : int  — nombre de recommandations

    Returns
    -------
    dict :
        anomaly_prob, risk_label, risk_proba, risk_probabilities,
        fault_label, fault_proba, fault_probabilities,
        ml_risk_score, recommendations
    """

    # ── Étape 1 : Anomaly detection ──────────────────────────
    anomaly_prob = get_anomaly_prob(obd2_data)
    print(f"\n🔴 Anomaly probability  : {anomaly_prob:.3f}")

    # ── Étape 2 : Risk level par composant ───────────────────
    vehicle_data["component"] = component
    risk_label, risk_proba, risk_index, risk_all = get_risk_level_and_proba(vehicle_data)
    print(f"⚠️  Risk level          : {risk_label} (confidence {risk_proba:.3f})")

    # ── Étape 3 : Fault type par composant ───────────────────
    fault_label, fault_proba, fault_all = get_fault_type_and_proba(vehicle_data)
    print(f"🔧 Fault type          : {fault_label} (confidence {fault_proba:.3f})")

    # ── Étape 4 : ML Risk Score composite ────────────────────
    ml_risk_score = compute_ml_risk_score(
        anomaly_prob, risk_label, fault_label, fault_proba, n_risk_classes=4
    )
    print(f"📊 ML Risk Score       : {ml_risk_score:.3f}")

    # Enrichit la requête TF-IDF avec TOUTES les descriptions de symptômes
    # associées au fault prédit (mapping 1-to-many) pour maximiser la similarité
    # avec les cas historiques dont les problem_description varient.
    fault_problem_labels = FAULT_TO_PROBLEMS.get(fault_label.lower(), [])
    fault_hint_str = " ".join(fault_problem_labels) if fault_problem_labels else fault_label.replace("_", " ")

    similar_cases = get_similar_cases(
        component=component,
        problem_description=problem_description,
        fault_label=fault_hint_str,
        ecu_data=ecu_data,
        service_history=service_history,
        top_k=max(top_k * 10, 50),
    )

    # ── Étape 6 : Score fusion par cas ───────────────────────
    order = SEMANTIC_RISK_ORDER.get(risk_label.lower().strip(), 1)
    risk_level_actuel = order / 3.0

    recommendations = []
    for _, case in similar_cases.iterrows():
        result_score  = encode_result(case.get("results", ""))
        repair_score  = encode_repair_status(case.get("repair_status", ""))
        specificity   = solution_specificity(case.get("solution_used", ""))
        case_risk_norm = (case.get("risk_score_num", 2) - 1) / 3.0
        risk_align    = 1.0 - abs(case_risk_norm - risk_level_actuel)

        # Bonus fault : concordance panne prédite ↔ problem_description du cas
        # (la colonne problem_description contient directement le type de panne)
        fmatch = fault_type_match_bonus(fault_label, case.get("problem_description", ""))

        base_score = fuse_scores(
            similarity=case["similarity_score"],
            ml_risk=ml_risk_score,
            case_result_score=result_score,
            repair_status_score=repair_score,
            risk_alignment=risk_align,
            specificity_score=specificity,
            risk_level_actuel=risk_level_actuel,
            fault_match=fmatch,
        )

        # Bonus si le problème textuel coïncide
        pmatch = problem_match_bonus(
            problem_description, case.get("problem_description", "")
        )
        final_score = float(np.clip(base_score * (1.0 + 0.15 * pmatch), 0.0, 1.0))

        urgency = (
            "🔴 CRITICAL" if final_score >= 0.75
            else "🟠 HIGH"   if final_score >= 0.55
            else "🟡 MEDIUM" if final_score >= 0.35
            else "🟢 LOW"
        )

        recommendations.append({
            "rank":                None,
            "final_score":         round(final_score, 4),
            "similarity_score":    round(case["similarity_score"], 4),
            "fault_match_score":   round(fmatch, 4),
            "ml_risk_score":       round(ml_risk_score, 4),
            "urgency":             urgency,
            "component":           case.get("component", ""),
            "problem_description": case.get("problem_description", ""),
            "diagnosis":           case.get("diagnosis", ""),
            "action":              case.get("how_to_fix_the_problem", ""),
            "solution_used":       case.get("solution_used", ""),
            "repair_status":       case.get("repair_status", ""),
            "results":             case.get("results", ""),
            "car_name":            case.get("car_name", ""),
            "confidence":          round(risk_proba, 4),
        })

    # ── Étape 7 : Tri + déduplication ────────────────────────
    recommendations.sort(key=lambda x: x["final_score"], reverse=True)

    # ── Cohérence diagnostic ↔ action ─────────────────────────
    # Le dataset associe parfois un diagnostic (ex: "Fuel pump issue") à une
    # action non liée (ex: "Replace spark plugs"). On calcule la cohérence
    # sémantique entre diagnosis et action et on pénalise les cas incohérents.
    # Mots-clés : si aucun mot du diagnostic n'apparaît dans l'action ET que
    # les deux textes ne partagent aucun token de composant, on pénalise.
    def _diag_action_coherence(diag: str, action: str) -> float:
        d_tokens = set(re.sub(r"[^a-z ]", "", diag.lower()).split())
        a_tokens = set(re.sub(r"[^a-z ]", "", action.lower()).split())
        # Stopwords à ignorer
        stop = {"the","a","an","and","or","of","to","in","is","it","for",
                "with","by","from","issue","failure","problem","fault","wear",
                "replaced","replace","repair","check","inspect","clean"}
        d_tokens -= stop
        a_tokens -= stop
        if not d_tokens:
            return 0.5   # neutre si vide
        overlap = len(d_tokens & a_tokens) / len(d_tokens)
        return float(np.clip(0.3 + overlap * 0.7, 0.0, 1.0))  # plancher 0.3

    # Appliquer le score de cohérence : pénalise final_score si incohérent
    for r in recommendations:
        coh = _diag_action_coherence(r["diagnosis"], r["action"])
        # Pénalité douce : réduit le score de max 20% si coh=0.3
        r["coherence"]   = round(coh, 3)
        r["final_score"] = round(r["final_score"] * (0.8 + 0.2 * coh), 4)

    # Re-trier après pénalité de cohérence
    recommendations.sort(key=lambda x: x["final_score"], reverse=True)

    def _action_target(action: str) -> str:
        """
        Extrait le 'composant cible' d'une action pour déduplication sémantique.
        'Replace spark plugs' et 'Replace faulty spark plugs' → 'spark plug'
        'Replace fuel pump' et 'Inspect fuel pump pressure' → 'fuel pump'
        Supprime : verbe initial, stopwords, adjectifs qualificatifs, pluriels.
        """
        a = action.lower().strip()
        # Supprimer verbe initial (premier mot)
        a = re.sub(r"^\w+\s+", "", a)
        # Supprimer stopwords et adjectifs courants
        a = re.sub(r"\b(the|a|an|faulty|worn|damaged|all|both|defective|and|or"
                   r"|leaking|clogged|dirty|old|new|broken|stuck|failed)\b", "", a)
        # Singulariser
        a = re.sub(r"s\b", "", a)
        a = re.sub(r"\s+", " ", a).strip()
        return a

    seen_targets: set = set()
    deduped: list     = []
    for r in recommendations:
        target = _action_target(r["action"])
        if target not in seen_targets:
            seen_targets.add(target)
            deduped.append(r)
        if len(deduped) >= top_k:
            break

    # Compléter si besoin (même verbe, objet différent)
    if len(deduped) < top_k:
        for r in recommendations:
            target = _action_target(r["action"])
            if target not in seen_targets:
                seen_targets.add(target)
                deduped.append(r)
            if len(deduped) >= top_k:
                break

    for i, r in enumerate(deduped[:top_k]):
        r["rank"] = i + 1

    return {
        "anomaly_prob":        round(anomaly_prob, 4),
        "risk_label":          risk_label,
        "risk_proba":          round(risk_proba, 4),
        "risk_probabilities":  risk_all,
        "fault_label":         fault_label,            # ← NOUVEAU
        "fault_proba":         round(fault_proba, 4),  # ← NOUVEAU
        "fault_probabilities": fault_all,              # ← NOUVEAU
        "ml_risk_score":       round(ml_risk_score, 4),
        "recommendations":     deduped[:top_k],
    }


# ─────────────────────────────────────────────────────────────
# 8. COÛT ESTIMÉ
# ─────────────────────────────────────────────────────────────

COST_ESTIMATES = {
    "spark plug":          (50,   150),
    "ignition coil":       (150,  400),
    "timing belt":         (300,  900),
    "fuel pump":           (200,  600),
    "coolant":             (100,  300),
    "thermostat":          (80,   250),
    "radiator":            (300,  900),
    "water pump":          (250,  700),
    "battery":             (100,  300),
    "alternator":          (300,  700),
    "brake":               (150,  500),
    "catalytic converter": (500, 1500),
    "oxygen sensor":       (150,  400),
    "maf sensor":          (100,  350),
    "egr valve":           (150,  500),
    "turbo":               (800, 2500),
    "head gasket":        (1000, 2500),
    "transmission":       (1500, 4000),
    "oil change":           (50,  120),
    "air filter":           (20,   60),
    "belt":                (100,  400),
}

def estimate_cost(action: str) -> str:
    action_lower = action.lower()
    for keyword, (lo, hi) in COST_ESTIMATES.items():
        if keyword in action_lower:
            return f"TND{lo}–TND{hi}"
    return "TND100–TND500"


# ─────────────────────────────────────────────────────────────
# 9. GÉNÉRATION DU RAISONNEMENT VIA CLAUDE API
# ─────────────────────────────────────────────────────────────

import urllib.request
import json as _json


def generate_reasoning(
    component: str,
    problem_description: str,
    diagnosis: str,
    action: str,
    anomaly_prob: float,
    risk_label: str,
    fault_label: str,
    ml_risk_score: float,
    ecu_data: str = "",
) -> str:
    """
    Génère un raisonnement clinique via Claude Sonnet.
    Intègre le fault_label prédit pour plus de précision.
    En cas d'échec API, le fallback rule-based exploite le diagnostic
    et le type de panne pour produire une explication cohérente.
    """
    prompt = (
        "You are an expert automotive diagnostic AI. "
        "Given the following vehicle situation, write ONE concise sentence (max 25 words) "
        "explaining WHY this specific action is recommended. "
        "Be technical and specific. No preamble.\n\n"
        f"Component: {component}\n"
        f"Problem: {problem_description}\n"
        f"Predicted fault type: {fault_label}\n"
        f"Diagnosis: {diagnosis}\n"
        f"Recommended action: {action}\n"
        f"ECU codes: {ecu_data or 'none'}\n"
        f"Risk level: {risk_label} (ML score: {ml_risk_score:.2f})\n"
        f"Anomaly probability: {anomaly_prob:.2f}\n"
    )
    try:
        payload = _json.dumps({
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 80,
            "messages": [{"role": "user", "content": prompt}],
        }).encode()
        req = urllib.request.Request(
            "https://api.anthropic.com/v1/messages",
            data=payload,
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = _json.loads(resp.read())
            return data["content"][0]["text"].strip()
    except Exception:
        # Fallback rule-based : relie directement l'action au fault_label prédit
        # et aux données OBD2 — ne mentionne PAS le diagnosis du cas historique
        # car il appartient à un autre véhicule et crée une confusion sémantique.
        fault_readable = fault_label.replace("_", " ") if fault_label != "normal" else component
        action_short   = str(action).strip()[:70]
        risk_short     = risk_label.replace("_", " ")
        ecu_str        = f" (codes: {ecu_data})" if ecu_data else ""

        return (
            f"ML model predicts {fault_readable} on {component}{ecu_str} — "
            f"{action_short} addresses this failure mode "
            f"({risk_short}, {anomaly_prob:.0%} anomaly probability)."
        )


def enrich_recommendations(
    result: dict,
    component: str,
    problem_description: str,
    ecu_data: str = "",
) -> dict:
    """Ajoute estimated_cost et reasoning à chaque recommandation."""
    print("\n🤖 Generating reasoning via Claude API...")
    for rec in result["recommendations"]:
        rec["estimated_cost"] = estimate_cost(rec["action"])
        rec["reasoning"] = generate_reasoning(
            component=component,
            problem_description=problem_description,
            diagnosis=rec["diagnosis"],
            action=rec["action"],
            anomaly_prob=result["anomaly_prob"],
            risk_label=result["risk_label"],
            fault_label=result["fault_label"],    # ← NOUVEAU
            ml_risk_score=result["ml_risk_score"],
            ecu_data=ecu_data,
        )
    return result


# ─────────────────────────────────────────────────────────────
# 10. AFFICHAGE
# ─────────────────────────────────────────────────────────────

def display_recommendations(result: dict):
    print("\n" + "=" * 65)
    print("🚗  VEHICLE DIAGNOSTIC RECOMMENDATIONS")
    print("=" * 65)
    print(f"  Anomaly probability  : {result['anomaly_prob']:.3f}")
    print(f"  Predicted risk level : {result['risk_label']} "
          f"(confidence {result['risk_proba']:.3f})")
    print(f"  Risk probabilities   : {result['risk_probabilities']}")
    print(f"  Predicted fault type : {result['fault_label']} "
          f"(confidence {result['fault_proba']:.3f})")
    print(f"  Fault probabilities  : {result['fault_probabilities']}")
    print(f"  ML Risk Score        : {result['ml_risk_score']:.3f}")
    print("=" * 65)

    for rec in result["recommendations"]:
        print(f"\n  Rank #{rec['rank']}  {rec['urgency']}")
        print(f"  {'─' * 58}")
        print(f"  Action       : {rec['action']}")
        print(f"  Confidence   : {rec['confidence']:.1%}  "
              f"(score {rec['final_score']:.4f})")
        print(f"  Fault match  : {rec['fault_match_score']:.3f}")
        print(f"  Est. Cost    : {rec.get('estimated_cost', 'N/A')}")
        print(f"  Reasoning    : {rec.get('reasoning', '')}")
        print(f"  ── Details ──────────────────────────────────────────")
        print(f"  Component    : {rec['component']}")
        print(f"  Problem      : {rec['problem_description']}")
        print(f"  Diagnosis    : {rec['diagnosis']}")
        print(f"  Solution     : {rec['solution_used']}")
        print(f"  Status       : {rec['repair_status']}  |  "
              f"Result: {rec['results']}")


# ─────────────────────────────────────────────────────────────
# 11. HELPER : contexte auto depuis CSV
# ─────────────────────────────────────────────────────────────

def build_context_from_data(obd2_row: dict, vehicle_row: dict) -> dict:
    """
    Dérive automatiquement les champs textuels nécessaires à recommend()
    à partir des deux lignes CSV. Aucune saisie manuelle requise.
    """
    component = str(vehicle_row.get("component", "engine")).lower().strip()

    dtc_fields = [k for k in vehicle_row
                  if k.startswith("dtc_code") or k.startswith("fault_code")]
    ecu_data = " ".join(
        str(vehicle_row[k]) for k in dtc_fields
        if pd.notna(vehicle_row.get(k))
    )

    problems = []
    coolant = float(obd2_row.get("coolant_temp_c", 0) or 0)
    if coolant > 100:
        problems.append("engine overheating")

    voltage = float(obd2_row.get("control_module_voltage_v", 14) or 14)
    if voltage < 12.0:
        problems.append("low battery voltage")

    rpm   = float(obd2_row.get("engine_rpm", 0) or 0)
    speed = float(obd2_row.get("vehicle_speed_kmh", 1) or 1)
    if speed > 0 and rpm / speed > 60:
        problems.append("abnormal RPM-speed ratio")

    load = float(obd2_row.get("engine_load_pct", 0) or 0)
    if load > 85:
        problems.append("high engine load")

    fuel_trim = float(obd2_row.get("short_fuel_trim_pct", 0) or 0)
    if abs(fuel_trim) > 10:
        problems.append("abnormal fuel trim")

    dtc_7d = int(vehicle_row.get("dtc_count_7d", 0) or 0)
    if dtc_7d > 0:
        problems.append(f"{dtc_7d} DTC fault codes in last 7 days")

    problem_description = (
        ", ".join(problems) if problems else f"{component} anomaly detected"
    )
    if ecu_data:
        problem_description += f" ({ecu_data})"

    overdue = []
    if float(vehicle_row.get("km_since_oil_change",     0) or 0) > 8000:
        overdue.append("oil change overdue")
    if float(vehicle_row.get("km_since_brake_service",  0) or 0) > 40000:
        overdue.append("brake service overdue")
    if float(vehicle_row.get("km_since_battery_change", 0) or 0) > 30000:
        overdue.append("battery check overdue")
    if float(vehicle_row.get("km_since_coolant_flush",  0) or 0) > 50000:
        overdue.append("coolant flush overdue")
    service_history = ", ".join(overdue) if overdue else "maintenance up to date"

    return {
        "component":           component,
        "problem_description": problem_description,
        "ecu_data":            ecu_data,
        "service_history":     service_history,
    }


# ─────────────────────────────────────────────────────────────
# 12. SEUILS DE DÉCLENCHEMENT
# ─────────────────────────────────────────────────────────────

ANOMALY_THRESHOLD = 0.5
# Une panne non-normale peut aussi déclencher les recs même si risk est médium
FAULT_TRIGGER_LABELS = {"high_risk", "critical_risk", "high", "critical"}


def should_trigger(
    anomaly_prob: float,
    risk_label: str,
    fault_label: str,
) -> tuple[bool, str]:
    """
    Décide si le moteur de recommandation doit se déclencher.

    Règles :
      1. Anomalie faible → pas de recommandation
      2. Anomalie + risque élevé/critique → recommandation
      3. Anomalie + panne réelle détectée (fault ≠ normal) → recommandation
         même si le risk_label seul n'est pas critique
    """
    if anomaly_prob <= ANOMALY_THRESHOLD:
        return False, (
            f"✅ No anomaly detected (prob={anomaly_prob:.3f} ≤ {ANOMALY_THRESHOLD}). "
            "No recommendation triggered."
        )

    high_risk   = risk_label.lower().strip() in FAULT_TRIGGER_LABELS
    fault_found = fault_label.lower() != "normal"

    if high_risk:
        return True, (
            f"🚨 ALERT — anomaly confirmed + high/critical risk ({risk_label}). "
            "Triggering recommendation engine..."
        )
    if fault_found:
        return True, (
            f"🚨 ALERT — anomaly confirmed + active fault detected ({fault_label}). "
            "Triggering recommendation engine..."
        )

    return False, (
        f"✅ Anomaly detected but risk is '{risk_label}' and no specific fault found. "
        "Continue monitoring."
    )


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":

    # ── Lecture des CSV ──────────────────────────────────────
    obd2_df    = pd.read_csv("01_anomaly_detection_obd2_readings.csv")
    feature_df = pd.read_csv("02_failure_classification_features.csv")

    VEHICLE_ID  = obd2_df["vehicle_id"].iloc[0]
    print(f"\n🚗 Running pipeline for vehicle_id: {VEHICLE_ID}")

    obd2_row    = obd2_df[obd2_df["vehicle_id"] == VEHICLE_ID].iloc[-1].to_dict()
    vehicle_row = feature_df[feature_df["vehicle_id"] == VEHICLE_ID].iloc[-1].to_dict()

    obd2_data    = {k: obd2_row[k] for k in OBD2_FEATURES if k in obd2_row}
    vehicle_data = vehicle_row.copy()

    ctx = build_context_from_data(obd2_row, vehicle_row)
    print(f"  Component        : {ctx['component']}")
    print(f"  Problem detected : {ctx['problem_description']}")
    print(f"  ECU data         : {ctx['ecu_data'] or 'none'}")
    print(f"  Service history  : {ctx['service_history']}")

    # ── Décision de déclenchement ────────────────────────────
    anomaly_prob = get_anomaly_prob(obd2_data)
    vehicle_data["component"] = ctx["component"]
    risk_label, risk_proba, _, _ = get_risk_level_and_proba(vehicle_data)
    fault_label, fault_proba, _  = get_fault_type_and_proba(vehicle_data)

    print(f"\n  Anomaly probability : {anomaly_prob:.3f}")
    print(f"  Risk level          : {risk_label} (confidence {risk_proba:.3f})")
    print(f"  Fault type          : {fault_label} (confidence {fault_proba:.3f})")

    trigger, message = should_trigger(anomaly_prob, risk_label, fault_label)
    print(f"\n{message}")

    if trigger:
        result = recommend(
            obd2_data=obd2_data,
            vehicle_data=vehicle_data,
            component=ctx["component"],
            problem_description=ctx["problem_description"],
            ecu_data=ctx["ecu_data"],
            service_history=ctx["service_history"],
            top_k=5,
        )
        result = enrich_recommendations(
            result,
            component=ctx["component"],
            problem_description=ctx["problem_description"],
            ecu_data=ctx["ecu_data"],
        )
        display_recommendations(result)

        rec_out = pd.DataFrame(result["recommendations"])
        rec_out.to_csv("recommendations_output.csv", index=False)
        print("\n💾 Recommendations saved to recommendations_output.csv")


    # ═══════════════════════════════════════════════════════
    # TEST MANUEL
    # ═══════════════════════════════════════════════════════

    test_obd2_data = {
        "engine_rpm":               4200,
        "vehicle_speed_kmh":          80,
        "coolant_temp_c":            108,
        "intake_air_temp_c":          45,
        "maf_airflow_gs":           12.5,
        "throttle_position_pct":      78,
        "control_module_voltage_v": 11.8,
        "engine_load_pct":            88,
        "short_fuel_trim_pct":        15,
        "long_fuel_trim_pct":         10,
    }

    test_vehicle_data = {
        "component":               "engine",
        "mileage_km":              145000,
        "vehicle_age_years":            8,
        "km_since_oil_change":      12000,
        "km_since_brake_service":   50000,
        "km_since_battery_change":  40000,
        "km_since_coolant_flush":   60000,
        "avg_rpm_30d":               3500,
        "std_rpm_30d":                450,
        "avg_throttle_30d":            70,
        "driving_aggression_score":  0.85,
        "dtc_count_7d":                 3,
        "dtc_count_30d":                8,
        "dtc_count_90d":               15,
        "total_engine_hours":        2200,
        "avg_coolant_temp_30d":       102,
        "max_coolant_temp_30d":       110,
        "avg_voltage_30d":           12.1,
        "min_voltage_30d":           11.6,
        "avg_speed_30d":               75,
        "fuel_trim_mean_30d":          13,
        "fuel_trim_std_30d":          6.5,
        "rpm_speed_ratio":             56,
        "coolant_delta_30d":            8,
        "voltage_drop_30d":           0.5,
        "idle_time_pct":              0.12,
        "highway_pct":                0.45,
        "year":                      2016,
    }

    test_ctx = {
        "component":           "engine",
        "problem_description": (
            "engine overheating, low voltage, high load, multiple DTC codes detected"
        ),
        "ecu_data":       "P0300 P0171 P0562",
        "service_history": "oil change overdue, coolant flush overdue",
    }

    print("\n\n================= 🧪 TEST RUN =================")

    anomaly_prob  = get_anomaly_prob(test_obd2_data)
    risk_label, risk_proba, _, risk_all  = get_risk_level_and_proba(test_vehicle_data)
    fault_label, fault_proba, fault_all  = get_fault_type_and_proba(test_vehicle_data)

    print("\n📊 Test Predictions:")
    print(f"  Anomaly Probability  : {anomaly_prob:.3f}")
    print(f"  Risk Label           : {risk_label} (confidence {risk_proba:.3f})")
    print(f"  Risk Probabilities   : {risk_all}")
    print(f"  Fault Type           : {fault_label} (confidence {fault_proba:.3f})")
    print(f"  Fault Probabilities  : {fault_all}")

    trigger, message = should_trigger(anomaly_prob, risk_label, fault_label)
    print(f"\n{message}")

    if trigger:
        result = recommend(
            obd2_data=test_obd2_data,
            vehicle_data=test_vehicle_data,
            component=test_ctx["component"],
            problem_description=test_ctx["problem_description"],
            ecu_data=test_ctx["ecu_data"],
            service_history=test_ctx["service_history"],
            top_k=5,
        )
        result = enrich_recommendations(
            result,
            component=test_ctx["component"],
            problem_description=test_ctx["problem_description"],
            ecu_data=test_ctx["ecu_data"],
        )
        display_recommendations(result)
    else:
        print("\n✅ Vehicle is safe — no recommendation triggered.")

C:\Users\lenovo2\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


📦 Loading saved models...
✅ All models loaded

📂 Loading recommendation dataset...
✅ 10000 historical cases loaded

🔍 Building TF-IDF index...
✅ TF-IDF matrix: (10000, 1017)

🚗 Running pipeline for vehicle_id: VH-0001
  Component        : brakes
  Problem detected : 1 DTC fault codes in last 7 days
  ECU data         : none
  Service history  : maintenance up to date

  Anomaly probability : 0.429
  Risk level          : low_risk (confidence 0.835)
  Fault type          : normal (confidence 0.991)

✅ No anomaly detected (prob=0.429 ≤ 0.5). No recommendation triggered.


================= 🧪 TEST RUN =================

📊 Test Predictions:
  Anomaly Probability  : 0.639
  Risk Label           : high_risk (confidence 0.898)
  Risk Probabilities   : {'high_risk': 0.898, 'low_risk': 0.0317, 'medium_risk': 0.0391, 'no_risk': 0.0312}
  Fault Type           : engine_misfire (confidence 0.395)
  Fault Probabilities  : {'engine_misfire': 0.3954, 'engine_overheating': 0.3054, 'normal': 0.2992}

🚨 